In [6]:
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

import pandas as pd
import plotly.graph_objs as go
import plotly.express as px

In [9]:
data = pd.read_csv('automobile_sales.csv')
data.head()

,Date,Year,Month,Recession,Consumer_Confidence,Seasonality_Weight,Price,Advertising_Expenditure,Competition,GDP,Growth_Rate,unemployment_rate,Automobile_Sales,Vehicle_Type,City
0,1980-01-31,1980,Jan,1,108.24,0.45,27704,1417.5,7,60.22,0.01,5.4,220.0,SmallFamilyCar,Georgia
1,1980-01-31,1980,Jan,1,108.24,0.45,77270,763.7,7,60.22,0.01,5.4,72.0,Sports,Georgia
2,1980-01-31,1980,Jan,1,108.24,0.36,19665,1417.5,7,60.22,0.01,5.4,238.0,SuperMiniCar,Georgia
3,1980-01-31,1980,Jan,1,108.24,0.38,36986,1417.5,7,60.22,0.01,5.4,224.0,MediumFamilyCar,Georgia
4,1980-02-29,1980,Feb,1,98.75,0.46,26609,2773.4,4,45.99,-0.31,4.8,280.0,SmallFamilyCar,New York


In [10]:
app = dash.Dash(__name__)
app.title = "Automobile Statistics Dashboard"

In [11]:
# Create dropdown menu options
dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
]

# List of years
year_list = [i for i in range(1980, 2024, 1)]

# Create the layout of the app
app.layout = html.Div([

    # TASK 2.1: Dashboard title
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 24
        }
    ),

    # TASK 2.2: Report type dropdown
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=dropdown_options,
            value='Select Statistics',
            placeholder='Select a report type',
            style={
                'width': '80%',
                'padding': '3px',
                'font-size': '20px',
                'text-align-last': 'center'
            }
        )
    ]),

    # TASK 2.2: Year dropdown
    html.Div([
        html.Label("Select Year:"),
        dcc.Dropdown(
            id='select-year',
            options=[{'label': i, 'value': i} for i in year_list],
            value='Select-year',
            placeholder='Select-year',
            style={
                'width': '80%',
                'padding': '3px',
                'font-size': '20px',
                'text-align-last': 'center'
            }
        )
    ]),

    # TASK 2.3: Output container
    html.Div([
        html.Div(
            id='output-container',
            className='chart-grid',
            style={'display': 'flex'}
        )
    ])
])

In [12]:
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value')
)
def update_input_container(selected_statistics):

    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True

In [13]:
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [
        Input(component_id='dropdown-statistics', component_property='value'),
        Input(component_id='select-year', component_property='value')
    ]
)
def update_output_container(selected_statistics, input_year):

    if selected_statistics == 'Recession Period Statistics':

        recession_data = data[data['Recession'] == 1]

        # Plot 1: Automobile sales over recession years
        yearly_rec = (
            recession_data.groupby('Year')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales During Recession Periods'
            )
        )

        # Plot 2: Average sales by vehicle type
        average_sales = (
            recession_data.groupby('Vehicle_Type')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Automobile Sales by Vehicle Type During Recession'
            )
        )

        # Plot 3: Advertising expenditure by vehicle type
        exp_rec = (
            recession_data.groupby('Vehicle_Type')['Advertising_Expenditure']
            .sum()
            .reset_index()
        )

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertising Expenditure by Vehicle Type During Recession'
            )
        )

        # Plot 4: Unemployment rate effect on sales
        unemployment_data = (
            recession_data.groupby(
                ['unemployment_rate', 'Vehicle_Type']
            )['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemployment_data,
                x='unemployment_rate',
                y='Automobile_Sales',
                color='Vehicle_Type',
                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        return [
            html.Div(
                className='chart-item',
                children=[html.Div(children=R_chart1), html.Div(children=R_chart2)],
                style={'display': 'flex'}
            ),
            html.Div(
                className='chart-item',
                children=[html.Div(children=R_chart3), html.Div(children=R_chart4)],
                style={'display': 'flex'}
            )
        ]

In [14]:
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [
        Input(component_id='dropdown-statistics', component_property='value'),
        Input(component_id='select-year', component_property='value')
    ]
)
def update_output_container(selected_statistics, input_year):

    # -------------------------
    # Recession Report
    # -------------------------
    if selected_statistics == 'Recession Period Statistics':

        recession_data = data[data['Recession'] == 1]

        yearly_rec = (
            recession_data.groupby('Year')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales During Recession Periods'
            )
        )

        average_sales = (
            recession_data.groupby('Vehicle_Type')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Automobile Sales by Vehicle Type During Recession'
            )
        )

        exp_rec = (
            recession_data.groupby('Vehicle_Type')['Advertising_Expenditure']
            .sum()
            .reset_index()
        )

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertising Expenditure by Vehicle Type During Recession'
            )
        )

        unemployment_data = (
            recession_data.groupby(
                ['unemployment_rate', 'Vehicle_Type']
            )['Automobile_Sales']
            .mean()
            .reset_index()
        )

        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemployment_data,
                x='unemployment_rate',
                y='Automobile_Sales',
                color='Vehicle_Type',
                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        return [
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart1),
                    html.Div(children=R_chart2)
                ],
                style={'display': 'flex'}
            ),
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart3),
                    html.Div(children=R_chart4)
                ],
                style={'display': 'flex'}
            )
        ]

    # -------------------------
    # Yearly Report
    # -------------------------
    elif selected_statistics == 'Yearly Statistics' and input_year:

        yearly_data = data[data['Year'] == input_year]

        # Plot 1: Yearly automobile sales for whole period
        yas = (
            data.groupby('Year')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        Y_chart1 = dcc.Graph(
            figure=px.line(
                yas,
                x='Year',
                y='Automobile_Sales',
                title='Yearly Automobile Sales'
            )
        )

        # Plot 2: Monthly automobile sales for selected year
        mas = (
            yearly_data.groupby('Month')['Automobile_Sales']
            .sum()
            .reset_index()
        )

        Y_chart2 = dcc.Graph(
            figure=px.line(
                mas,
                x='Month',
                y='Automobile_Sales',
                title='Total Monthly Automobile Sales'
            )
        )

        # Plot 3: Average sales by vehicle type for selected year
        avr_vdata = (
            yearly_data.groupby('Vehicle_Type')['Automobile_Sales']
            .mean()
            .reset_index()
        )

        Y_chart3 = dcc.Graph(
            figure=px.bar(
                avr_vdata,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type'
            )
        )

        # Plot 4: Advertising expenditure by vehicle type
        exp_data = (
            yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure']
            .sum()
            .reset_index()
        )

        Y_chart4 = dcc.Graph(
            figure=px.pie(
                exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure for Each Vehicle'
            )
        )

        return [
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart1),
                    html.Div(children=Y_chart2)
                ],
                style={'display': 'flex'}
            ),
            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart3),
                    html.Div(children=Y_chart4)
                ],
                style={'display': 'flex'}
            )
        ]

In [15]:
app.run(debug=False)